## spark setup

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 19:06:51 WARN Utils: Your hostname, Alexs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.36 instead (on interface en0)
26/03/08 19:06:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 19:06:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. spark version

In [2]:
spark.version

'4.1.1'

## 2. yellow november 2025

In [5]:
!curl https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -o data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 67.8M  100 67.8M    0     0  67.5M      0  0:00:01  0:00:01 --:--:-- 67.5M


In [7]:
df = spark.read.parquet("data/yellow_tripdata_2025-11.parquet")
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [8]:
df.repartition(4).write.parquet("data/yellow_tripdata_2025-11-out.parquet")

In [11]:
!ls -althr data/yellow_tripdata_2025-11-out.parquet | grep .parquet

-rw-r--r--@  1 alexbenasutti  staff   195K Mar  8 19:08 .part-00002-10b99b86-5c18-402d-965b-656d36aa7c57-c000.snappy.parquet.crc
-rw-r--r--@  1 alexbenasutti  staff   195K Mar  8 19:08 .part-00001-10b99b86-5c18-402d-965b-656d36aa7c57-c000.snappy.parquet.crc
-rw-r--r--@  1 alexbenasutti  staff   195K Mar  8 19:08 .part-00000-10b99b86-5c18-402d-965b-656d36aa7c57-c000.snappy.parquet.crc
-rw-r--r--@  1 alexbenasutti  staff    24M Mar  8 19:08 part-00001-10b99b86-5c18-402d-965b-656d36aa7c57-c000.snappy.parquet
-rw-r--r--@  1 alexbenasutti  staff    24M Mar  8 19:08 part-00000-10b99b86-5c18-402d-965b-656d36aa7c57-c000.snappy.parquet
-rw-r--r--@  1 alexbenasutti  staff    24M Mar  8 19:08 part-00002-10b99b86-5c18-402d-965b-656d36aa7c57-c000.snappy.parquet
-rw-r--r--@  1 alexbenasutti  staff   195K Mar  8 19:08 .part-00003-10b99b86-5c18-402d-965b-656d36aa7c57-c000.snappy.parquet.crc
-rw-r--r--@  1 alexbenasutti  staff    24M Mar  8 19:08 part-00003-10b99b86-5c18-402d-965b-656d36aa7c57-c000.sna

## 3. count records

In [12]:
df.filter("to_date(tpep_pickup_datetime) = '2025-11-15'").count()

162604

## 4. longest trip

In [ ]:
import pyspark.sql.functions as sf

(
    df.withColumn(
        "trip_hours", 
        sf.round(
            sf.timestamp_diff("minute", "tpep_pickup_datetime", "tpep_dropoff_datetime") / 60, 1
        ),
    )
    .select("trip_hours")
    .orderBy("trip_hours", ascending=False)
    .limit(1)
    .show()
)

+----------+
|trip_hours|
+----------+
|      90.6|
+----------+



## 5. user interface

In [22]:
# localhost:4040

## 6. least frequent pickup location zone

In [23]:
!curl https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -o data/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12331  100 12331    0     0  78370      0 --:--:-- --:--:-- --:--:-- 78541


In [24]:
lookup = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True)
lookup.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [27]:
df.show(1)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [25]:
lookup.createOrReplaceTempView("lookup")
df.createOrReplaceTempView("trips")

In [28]:
spark.sql("""
    SELECT l.Zone, count(*) AS num_trips
    FROM trips AS t
    JOIN lookup AS l
        ON t.PULocationID = l.LocationID
    GROUP BY l.Zone
    ORDER BY num_trips ASC
""").show()

+--------------------+---------+
|                Zone|num_trips|
+--------------------+---------+
|Governor's Island...|        1|
|Eltingville/Annad...|        1|
|       Arden Heights|        1|
|       Port Richmond|        3|
|       Rikers Island|        4|
|   Rossville/Woodrow|        4|
| Green-Wood Cemetery|        4|
|         Great Kills|        4|
|         Jamaica Bay|        5|
|         Westerleigh|       12|
|        Crotona Park|       14|
|             Oakwood|       14|
|New Dorp/Midland ...|       14|
|       West Brighton|       14|
|       Willets Point|       15|
|Breezy Point/Fort...|       16|
|Saint George/New ...|       17|
|       Broad Channel|       18|
|     Mariners Harbor|       21|
|Heartland Village...|       22|
+--------------------+---------+
only showing top 20 rows
